# DeBERTa-v3-base PCL Training (Loads Augmented CSV)

This notebook trains/evaluates the DeBERTa classifier.

Expected augmentation artifact:
- `augmented_train_data.csv` (or `/kaggle/working/augmented_train_data.csv` on Kaggle)

The CSV is appended to the original train split before tokenization.


In [ ]:
!pip install -q huggingface_hub datasets transformers accelerate scikit-learn seaborn


In [ ]:
# Optional Hugging Face login (safe to skip if not needed)
from huggingface_hub import login
KAGGLE = False
if KAGGLE:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('HF token loaded from .env and login completed.')
else:
    from dotenv import load_dotenv
    
    load_dotenv()
    hf_token = os.getenv('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('HF token loaded from .env and login completed.')

In [ ]:
import os
import re
import random
import inspect

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 256
NUM_LABELS = 2

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

EVAL_EVERY = 20
LOG_EVERY = 50

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection'
else:
    DATA_ROOT = '..'

if os.path.exists('/kaggle/working'):
    AUGMENTED_INPUT_CSV = '/kaggle/working/augmented_train_data.csv'
else:
    AUGMENTED_INPUT_CSV = 'augmented_train_data.csv'

TSV_PATH = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')

OUTPUT_DIR = 'checkpoints/deberta_v3_augmented_trainer'
LOG_DIR = 'logs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DATA_ROOT           : {DATA_ROOT}')
print(f'AUGMENTED_INPUT_CSV : {AUGMENTED_INPUT_CSV}')
print(f'OUTPUT_DIR          : {OUTPUT_DIR}')


In [ ]:
def load_task1(train_path: str) -> pd.DataFrame:
    """
    Load Task 1 data and convert original labels to binary:
      0/1 -> 0 (No-PCL)
      2/3/4 -> 1 (PCL)
    """
    rows = []
    with open(train_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue

            par_id = parts[0]
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]
            text = parts[4]
            orig_label = parts[-1]
            label = 0 if orig_label in {'0', '1'} else 1

            rows.append(
                {
                    'par_id': str(par_id),
                    'art_id': art_id,
                    'keyword': keyword,
                    'country': country,
                    'text': text,
                    'label': label,
                    'orig_label': orig_label,
                }
            )

    return pd.DataFrame(
        rows,
        columns=['par_id', 'art_id', 'keyword', 'country', 'text', 'label', 'orig_label'],
    )


def preprocess_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)

print(f'Loaded dataset: {len(df):,} samples')
print(df['label'].value_counts().sort_index().rename({0: 'No-PCL', 1: 'PCL'}))


In [ ]:
# ============================================================
# Official Train/Dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df = pd.read_csv(DEV_IDS_PATH, dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)
dev_df = df[df['par_id'].isin(dev_par_ids)].copy().reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].copy().reset_index(drop=True)
if len(leftover_df) > 0:
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)
    print(f'Appended {len(leftover_df):,} unassigned samples to training set.')


In [ ]:
# ============================================================
# Load augmented CSV and append to original train split
# ============================================================
required_cols = ['par_id', 'art_id', 'keyword', 'country', 'text', 'clean_text', 'label', 'orig_label']

if os.path.exists(AUGMENTED_INPUT_CSV):
    aug_df = pd.read_csv(AUGMENTED_INPUT_CSV, dtype={'par_id': str})
    missing = [c for c in required_cols if c not in aug_df.columns]
    if missing:
        raise ValueError(f'Augmented CSV missing required columns: {missing}')

    aug_df = aug_df[required_cols].copy()
    aug_df['label'] = aug_df['label'].astype(int)
    train_final_df = pd.concat([train_df, aug_df], ignore_index=True)
    print(f'Loaded augmented CSV with {len(aug_df):,} rows from {AUGMENTED_INPUT_CSV}')
else:
    print(f'No augmented CSV found at {AUGMENTED_INPUT_CSV}. Training on original train split only.')
    aug_df = pd.DataFrame(columns=required_cols)
    train_final_df = train_df.copy()


def describe(name, frame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum()) if n else 0
    n_no = int((frame['label'] == 0).sum()) if n else 0
    frac = (n_pcl / n) if n > 0 else 0.0
    print(f'{name:<16} -> total={n:,} | PCL={n_pcl:,} ({frac:.1%}) | No-PCL={n_no:,}')


describe('Train(raw)', train_df)
describe('Augmented rows', aug_df)
describe('Train(final)', train_final_df)
describe('Dev', dev_df)


In [ ]:
# ============================================================
# Tokenization + Hugging Face Datasets
# ============================================================
train_hf = Dataset.from_pandas(
    train_final_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)
dev_hf = Dataset.from_pandas(
    dev_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


train_ds = train_hf.map(tokenize, batched=True, remove_columns=['text'])
dev_ds = dev_hf.map(tokenize, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
dev_ds = dev_ds.rename_column('label', 'labels')

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(dev_ds)


In [ ]:
# ============================================================
# Metrics
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_no_pcl': f1_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'f1_pcl': f1_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'prec_no_pcl': precision_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'prec_pcl': precision_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'rec_no_pcl': recall_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'rec_pcl': recall_score(labels, preds, pos_label=1, average='binary', zero_division=0),
    }


In [ ]:
# ============================================================
# Model + TrainingArguments + Trainer
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

training_args = TrainingArguments(
    RUN_NAME,
    num_train_epochs=2,
    save_total_limit=2,
    learning_rate=1e-5,
    eval_strategy='epoch',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    warmup_steps=2000,
    lr_scheduler_type='cosine',
    adam_epsilon=1e-6,
    weight_decay=0.01,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer configured.')


In [ ]:
train_result = trainer.train()
train_result


In [ ]:
eval_metrics = trainer.evaluate()
print('Evaluation metrics:')
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')


In [ ]:
# ============================================================
# Detailed evaluation report
# ============================================================
pred_output = trainer.predict(dev_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred No-PCL', 'Pred PCL'],
    yticklabels=['True No-PCL', 'True PCL'],
)
plt.title('Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()


In [ ]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, 'best')
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f'Saved best model and tokenizer to: {BEST_MODEL_DIR}')
